In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


In [3]:
# Load training data
df = pd.read_csv("/content/train_users (1).csv")

# Check data
print(df.shape)
print(df.head())
print(df.info())


(2000, 33)
  user_id   age  income  clicks  purchase_amount  session_duration  \
0   U7392   NaN   23053      10           500.00             17.34   
1   U2702  56.0   20239      11           913.33             22.22   
2   U2461   NaN   13907       9          1252.62             41.57   
3   U7475   NaN   26615      12           500.00             30.17   
4   U6040  32.0   27958      13           500.00             65.27   

   content_variety  engagement_score  num_transactions  avg_monthly_spend  \
0          0.36661          37.29781                 3             187.44   
1          0.61370          59.36342                 5             145.15   
2          0.80368          76.78706                 7             282.03   
3          0.26499          30.19441                10             195.35   
4          0.36385          37.12153                 5             439.68   

   ...  screen_brightness  battery_percentage  cart_abandonment_count  \
0  ...                4.0       

In [4]:
X = df.drop("label", axis=1)
y = df["label"]


In [5]:
num_cols = X.select_dtypes(include=['int64', 'float64']).columns
cat_cols = X.select_dtypes(include=['object']).columns


In [6]:
# Numerical → median
num_imputer = SimpleImputer(strategy="median")
X[num_cols] = num_imputer.fit_transform(X[num_cols])

# Categorical → most frequent
cat_imputer = SimpleImputer(strategy="most_frequent")
X[cat_cols] = cat_imputer.fit_transform(X[cat_cols])


In [7]:
label_encoders = {}

for col in cat_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])
    label_encoders[col] = le


In [8]:
y_encoder = LabelEncoder()
y = y_encoder.fit_transform(y)


In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [10]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [11]:
model = GaussianNB()
model.fit(X_train, y_train)


GaussianNB()

In [12]:
y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))


Accuracy: 0.83

Classification Report:
               precision    recall  f1-score   support

           0       0.85      0.83      0.84       142
           1       0.84      0.89      0.86       142
           2       0.80      0.75      0.77       116

    accuracy                           0.83       400
   macro avg       0.83      0.83      0.83       400
weighted avg       0.83      0.83      0.83       400


Confusion Matrix:
 [[118   6  18]
 [ 11 127   4]
 [ 10  19  87]]


Reasons for using GaussianNB :

The dataset contains mostly continuous numerical features (income, session duration, spend, etc.), and GaussianNB is designed for continuous data assuming a normal distribution.

It works well when there are many independent features, which fits this user-behavior dataset.

GaussianNB is fast and efficient, making it suitable for real-time use as a Context Detector in a bandit system.

It performs reliably even with moderate dataset size and high-dimensional data without heavy tuning.